# Efficient Frontier & Mean-Variance Portfolio Optimizer

Pulls historical prices for a basket of tickers, estimates annualized expected returns and a Ledoit-Wolf shrunk covariance matrix, then solves for the Max Sharpe and Global Minimum Variance portfolios via constrained optimization (SLSQP) and traces the efficient frontier. Also runs a Monte Carlo simulation for a visual reference cloud, plots the frontier with the Capital Allocation Line, and produces allocation and correlation charts.

Runs top-to-bottom in Colab; only `yfinance` and `fredapi` need installing.

In [ ]:

!pip install -q yfinance fredapi

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from google.colab import userdata
from fredapi import Fred

np.random.seed(42)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
})

print("Environment ready.")

In [ ]:
# Universe, lookback window, risk-free rate
TICKERS = ["SPY", "QQQ", "TLT", "GLD", "VNQ", "EFA", "IEF", "HYG"]
LOOKBACK_YEARS = 8
TRADING_DAYS_PER_YEAR = 252
N_MONTE_CARLO_PORTFOLIOS = 15000

END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.DateOffset(years=LOOKBACK_YEARS)

try:
    api_key = userdata.get('FRED_API_KEY')
    fred = Fred(api_key=api_key)
    live_rate = fred.get_series('DTB3').iloc[-1] / 100  # 3-month T-bill, secondary market
    RISK_FREE_RATE = live_rate
    print(f"Live RISK_FREE_RATE (FRED DTB3): {RISK_FREE_RATE:.4%}")
except Exception as e:
    RISK_FREE_RATE = 0.02
    print(f"Using default RISK_FREE_RATE (0.02) due to: {e}")

print(f"Universe: {TICKERS}")
print(f"Window:   {START_DATE.date()} to {END_DATE.date()}")

In [ ]:

def download_prices(tickers, start, end, max_retries=3):
    """Download adjusted close prices, retrying since yfinance batch
    calls occasionally drop tickers on the first attempt."""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            raw = yf.download(
                tickers,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
            )
            if raw.empty:
                raise ValueError("Empty dataframe returned by yfinance.")

            if isinstance(raw.columns, pd.MultiIndex):
                prices = pd.DataFrame({t: raw[t]["Close"] for t in tickers if t in raw.columns.get_level_values(0)})
            else:
                prices = raw[["Close"]].rename(columns={"Close": tickers[0]})

            missing = set(tickers) - set(prices.columns)
            if missing:
                print(f"Warning: no data returned for {missing}")

            return prices.sort_index()

        except Exception as e:
            last_error = e
            print(f"Attempt {attempt}/{max_retries} failed: {e}")

    raise RuntimeError(f"Failed to download price data after {max_retries} attempts: {last_error}")


prices = download_prices(TICKERS, START_DATE, END_DATE)
print(f"Downloaded {prices.shape[0]} trading days x {prices.shape[1]} tickers")
prices.tail()

In [ ]:
#Cleaning
def clean_prices(df, max_missing_frac=0.05):
    """Drop tickers with too much missing data, forward-fill small gaps,
    drop any remaining rows with NaNs."""
    missing_frac = df.isna().mean()
    bad_tickers = missing_frac[missing_frac > max_missing_frac].index.tolist()
    if bad_tickers:
        print(f"Dropping tickers with >{max_missing_frac:.0%} missing data: {bad_tickers}")
        df = df.drop(columns=bad_tickers)

    df = df.ffill().dropna()
    return df


def compute_returns_and_risk(df, trading_days=TRADING_DAYS_PER_YEAR):
    """Daily log returns, annualized mean, and Ledoit-Wolf shrunk covariance."""
    log_returns = np.log(df / df.shift(1)).dropna()

    mean_annual_returns = log_returns.mean() * trading_days

    lw = LedoitWolf()
    lw.fit(log_returns)
    cov_annual = pd.DataFrame(lw.covariance_, index=log_returns.columns, columns=log_returns.columns) * trading_days

    return log_returns, mean_annual_returns, cov_annual


prices_clean = clean_prices(prices)
log_returns, mu, cov_matrix = compute_returns_and_risk(prices_clean)

assets = list(prices_clean.columns)
n_assets = len(assets)

print("Annualized expected returns:")
print(mu.round(4))
print(f"\nCovariance matrix shape: {cov_matrix.shape}")

In [ ]:
# Portfolio stats and sim
def portfolio_performance(weights, mean_returns, cov):
    """(expected annual return, annual volatility, Sharpe ratio) for a weight vector."""
    weights = np.array(weights)
    port_return = np.dot(weights, mean_returns)
    port_vol = np.sqrt(weights.T @ cov @ weights)
    sharpe = (port_return - RISK_FREE_RATE) / port_vol if port_vol > 0 else np.nan
    return port_return, port_vol, sharpe


def monte_carlo_portfolios(mean_returns, cov, n_portfolios=N_MONTE_CARLO_PORTFOLIOS):
    """Sample long-only, fully-invested portfolios via Dirichlet weights
    and record their risk/return/Sharpe."""
    n = len(mean_returns)
    results = np.zeros((n_portfolios, 3))
    weight_records = np.zeros((n_portfolios, n))

    for i in range(n_portfolios):
        w = np.random.dirichlet(np.ones(n))
        ret, vol, sharpe = portfolio_performance(w, mean_returns, cov)
        results[i] = [ret, vol, sharpe]
        weight_records[i] = w

    return pd.DataFrame(results, columns=["Return", "Volatility", "Sharpe"]), weight_records


try:
    mc_results, mc_weights = monte_carlo_portfolios(mu, cov_matrix)
    print(f"Simulated {len(mc_results):,} random portfolios.")
    mc_results.describe()
except Exception as e:
    raise RuntimeError(f"Monte Carlo simulation failed: {e}")

In [ ]:
#Max Sharpe and Global Minimum Variance
def negative_sharpe(weights, mean_returns, cov):
    _, _, sharpe = portfolio_performance(weights, mean_returns, cov)
    return -sharpe


def portfolio_volatility(weights, mean_returns, cov):
    _, vol, _ = portfolio_performance(weights, mean_returns, cov)
    return vol


def solve_optimal_portfolio(objective_fn, mean_returns, cov, extra_constraints=None):
    """SLSQP solver: long-only (0 <= w <= 1), fully invested (sum w = 1).
    extra_constraints bolts on a target-return constraint for frontier sweeps."""
    n = len(mean_returns)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    if extra_constraints:
        constraints += extra_constraints

    bounds = tuple((0, 1) for _ in range(n))
    init_guess = np.repeat(1 / n, n)

    result = minimize(
        objective_fn,
        init_guess,
        args=(mean_returns, cov),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-10},
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed to converge: {result.message}")

    return result.x


try:
    max_sharpe_weights = solve_optimal_portfolio(negative_sharpe, mu, cov_matrix)
    min_vol_weights = solve_optimal_portfolio(portfolio_volatility, mu, cov_matrix)

    ms_ret, ms_vol, ms_sharpe = portfolio_performance(max_sharpe_weights, mu, cov_matrix)
    mv_ret, mv_vol, mv_sharpe = portfolio_performance(min_vol_weights, mu, cov_matrix)

    print("Max Sharpe Portfolio  -> Return: {:.2%}  Vol: {:.2%}  Sharpe: {:.2f}".format(ms_ret, ms_vol, ms_sharpe))
    print("Min Variance Portfolio -> Return: {:.2%}  Vol: {:.2%}  Sharpe: {:.2f}".format(mv_ret, mv_vol, mv_sharpe))
except Exception as e:
    raise RuntimeError(f"Portfolio optimization failed: {e}")

In [ ]:
#Frontier boundary
def efficient_frontier(mean_returns, cov, n_points=60):
    """Sweep target returns between the min and max achievable mean return,
    minimizing volatility at each target to trace the analytical frontier."""
    target_returns = np.linspace(mean_returns.min(), mean_returns.max(), n_points)
    frontier_vols = []

    for target in target_returns:
        constraint = [{"type": "eq", "fun": lambda w, t=target: np.dot(w, mean_returns) - t}]
        try:
            w = solve_optimal_portfolio(portfolio_volatility, mean_returns, cov, extra_constraints=constraint)
            _, vol, _ = portfolio_performance(w, mean_returns, cov)
            frontier_vols.append(vol)
        except RuntimeError:
            frontier_vols.append(np.nan)

    frontier_df = pd.DataFrame({"Return": target_returns, "Volatility": frontier_vols}).dropna()
    return frontier_df


frontier_df = efficient_frontier(mu, cov_matrix)
print(f"Solved {len(frontier_df)} points on the efficient frontier boundary.")

## Visualizations & performance summary

In [ ]:
# Frontier plot
fig, ax = plt.subplots(figsize=(11, 7))

sc = ax.scatter(
    mc_results["Volatility"], mc_results["Return"],
    c=mc_results["Sharpe"], cmap="viridis", s=8, alpha=0.35, label="Simulated portfolios"
)
cbar = fig.colorbar(sc)
cbar.set_label("Sharpe Ratio")

ax.plot(frontier_df["Volatility"], frontier_df["Return"], color="black", linewidth=2.2,
        label="Efficient Frontier (analytical)")

ax.scatter(ms_vol, ms_ret, color="crimson", marker="*", s=450, edgecolor="black",
           linewidth=0.8, label="Max Sharpe Portfolio", zorder=5)
ax.scatter(mv_vol, mv_ret, color="dodgerblue", marker="D", s=140, edgecolor="black",
           linewidth=0.8, label="Min Variance Portfolio", zorder=5)

# Capital Allocation
cal_x = np.linspace(0, mc_results["Volatility"].max(), 50)
cal_y = RISK_FREE_RATE + ms_sharpe * cal_x
ax.plot(cal_x, cal_y, linestyle="--", color="gray", linewidth=1.4, label="Capital Allocation Line")

ax.set_xlabel("Annualized Volatility (Risk)")
ax.set_ylabel("Annualized Expected Return")
ax.set_title("Efficient Frontier — Mean-Variance Optimization")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=150)
plt.show()

In [ ]:
# Allocation heatmap
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

def plot_allocation(ax, weights, title):
    w = pd.Series(weights, index=assets)
    w = w[w > 0.005]  # hide near-zero slivers
    ax.pie(w, labels=w.index, autopct="%1.1f%%", startangle=90,
           wedgeprops={"edgecolor": "white", "linewidth": 1})
    ax.set_title(title)

plot_allocation(axes[0], max_sharpe_weights, "Max Sharpe Allocation")
plot_allocation(axes[1], min_vol_weights, "Min Variance Allocation")

corr = log_returns.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[2],
            cbar_kws={"label": "Correlation"})
axes[2].set_title("Asset Correlation Matrix")

plt.tight_layout()
plt.savefig("allocation_and_correlation.png", dpi=150)
plt.show()

In [ ]:
# Performance metrics & export
def summarize_portfolio(weights, mean_returns, cov, label):
    ret, vol, sharpe = portfolio_performance(weights, mean_returns, cov)
    row = {"Portfolio": label, "Expected Return": ret, "Volatility": vol, "Sharpe Ratio": sharpe}
    row.update({f"Weight: {a}": w for a, w in zip(assets, weights)})
    return row

summary = pd.DataFrame([
    summarize_portfolio(max_sharpe_weights, mu, cov_matrix, "Max Sharpe"),
    summarize_portfolio(min_vol_weights, mu, cov_matrix, "Min Variance"),
])

display_cols = ["Portfolio", "Expected Return", "Volatility", "Sharpe Ratio"]
summary_display = summary[display_cols].copy()
summary_display["Expected Return"] = summary_display["Expected Return"].map("{:.2%}".format)
summary_display["Volatility"] = summary_display["Volatility"].map("{:.2%}".format)
summary_display["Sharpe Ratio"] = summary_display["Sharpe Ratio"].map("{:.2f}".format)

print("=" * 60)
print("PORTFOLIO PERFORMANCE SUMMARY")
print("=" * 60)
print(summary_display.to_string(index=False))

summary.to_csv("performance_summary.csv", index=False)
prices_clean.to_csv("prices_cache.csv")

print("\nSaved: efficient_frontier.png, allocation_and_correlation.png, performance_summary.csv, prices_cache.csv")

## Ideas to extend

- Black-Litterman blending instead of pure historical-mean returns (MVO is notoriously sensitive to noisy return inputs)
- Rolling re-optimization backtest to see realized vs. in-sample performance
- Transaction-cost / turnover constraints against a current holdings vector
- Streamlit/Dash wrapper for an interactive risk-tolerance slider